# V005 — Validate BCHH relative infrasound calibration against nearby 2016 launches

**Status:** validation/support notebook; **not** part of the production paper pipeline.

This notebook preserves the calibration cross-check used to support qualitative manuscript statements about the stability of the BCHH infrasound-channel relative gains around the September 2016 AMOS-6 failure.

It compares raw-count DD1/DD2/DD3 amplitudes for three routine launches close in time to the AMOS-6 event:

- JCSAT-16 — 2016-08-14
- AFSPC-6 — 2016-08-19
- OSIRIS-REx — 2016-09-08

and includes AMOS-6 as an extreme-amplitude **stress test**, not as a calibration-setting event.

The notebook tests **relative channel gain stability only**. It does not establish an absolute laboratory calibration, prove sensor serial-number assignments, or demonstrate linearity at the largest AMOS-6 amplitudes.

### Inputs

Preferred portable inputs are raw-count MiniSEED extracts in:

`data/miniseed/2016_launch_calibration/`

If an extract is absent, the notebook can create it from the local SDS archive configured in `local_config.toml`.

In [1]:
from pathlib import Path
import sys
from itertools import combinations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from obspy import Stream, UTCDateTime, read
from obspy.clients.filesystem.sds import Client as SDSClient

here = Path.cwd().resolve()
ROOT = next(
    (p for p in [here, *here.parents] if (p / "modules" / "project_config.py").is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError("Run from the Falcon 9 repository or a subdirectory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from modules import project_config as config

OUT = config.notebook_output_dir(5, "validate_bchh_relative_calibration")
OUT.mkdir(parents=True, exist_ok=True)
INPUT_DIR = config.MINISEED_DIR / "2016_launch_calibration"

PRESSURE_IDS = (
    "1R.BCHH.10.DD1",
    "1R.BCHH.10.DD2",
    "1R.BCHH.10.DD3",
)

# Empirical 2016 conversion factors already adopted by the production workflow.
# This validation notebook does not re-estimate them.
ADOPTED_COUNTS_PER_PA = {
    "DD1": 8.10,
    "DD2": 0.690,
    "DD3": 10.0,
}

EVENTS = pd.DataFrame([
    {
        "event_id": "jcsat16",
        "mission": "JCSAT-16",
        "event_time_utc": "2016-08-14T05:26:00Z",
        "role": "routine_launch",
        "file": "jcsat16_bchh_raw.mseed",
    },
    {
        "event_id": "afspc6",
        "mission": "AFSPC-6",
        "event_time_utc": "2016-08-19T04:52:00Z",
        "role": "routine_launch",
        "file": "afspc6_bchh_raw.mseed",
    },
    {
        "event_id": "amos6",
        "mission": "AMOS-6 fireball",
        "event_time_utc": "2016-09-01T13:07:15.5136Z",
        "role": "stress_test",
        "file": "amos6_bchh_raw.mseed",
    },
    {
        "event_id": "osirisrex",
        "mission": "OSIRIS-REx",
        "event_time_utc": "2016-09-08T23:05:00Z",
        "role": "routine_launch",
        "file": "osirisrex_bchh_raw.mseed",
    },
])

display(EVENTS)
print("Validation output:", OUT)

,event_id,mission,event_time_utc,role,file
0,jcsat16,JCSAT-16,2016-08-14T05:26:00Z,routine_launch,jcsat16_bchh_raw.mseed
1,afspc6,AFSPC-6,2016-08-19T04:52:00Z,routine_launch,afspc6_bchh_raw.mseed
2,amos6,AMOS-6 fireball,2016-09-01T13:07:15.5136Z,stress_test,amos6_bchh_raw.mseed
3,osirisrex,OSIRIS-REx,2016-09-08T23:05:00Z,routine_launch,osirisrex_bchh_raw.mseed


Validation output: /Users/thompsong/Library/CloudStorage/Box-Box/thompsong/3_Project_Documents/NASAprojects/201602_Rocket_Seismology/writing/falcon9_fireball_paper/outputs/005_validate_bchh_relative_calibration


## Processing choices

All comparisons are made in **raw digitizer counts**. Each trace is linearly detrended, demeaned, cosine tapered and zero-phase high-pass filtered at 0.1 Hz.

A common 50-s signal window is determined independently for each event using DD2/DD3 onset SNR. Pairwise RMS ratios are then measured within that event-specific window. Lagged correlation is retained only as a waveform-consistency diagnostic; the lag does not materially affect RMS ratios.

In [2]:
PRE_S, POST_S = 30.0, 240.0
HIGHPASS_HZ = 0.1
FILTER_CORNERS = 4
TAPER_FRACTION = 0.05
NOISE_WINDOW = (-25.0, -5.0)
SEARCH_WINDOW = (0.0, 180.0)
ENV_WINDOW_S = 1.0
ONSET_SNR = 5.0
CAL_WINDOW_LENGTH_S = 50.0


def preprocess_counts(st):
    out = Stream()
    for tr in st:
        if tr.id not in PRESSURE_IDS:
            continue
        x = tr.copy()
        x.detrend("linear")
        x.detrend("demean")
        x.taper(max_percentage=TAPER_FRACTION, type="cosine")
        x.filter(
            "highpass",
            freq=HIGHPASS_HZ,
            corners=FILTER_CORNERS,
            zerophase=True,
        )
        out.append(x)
    return out


def rolling_rms(x, n):
    values = np.asarray(x, dtype=float)
    return np.sqrt(
        pd.Series(values**2)
        .rolling(n, center=True, min_periods=max(1, n // 3))
        .mean()
        .to_numpy()
    )


def channel_snr(tr, t0):
    sr = float(tr.stats.sampling_rate)
    t = np.arange(tr.stats.npts) / sr + float(tr.stats.starttime - t0)
    env = rolling_rms(tr.data, max(1, round(ENV_WINDOW_S * sr)))
    mask = (t >= NOISE_WINDOW[0]) & (t <= NOISE_WINDOW[1])
    noise = np.nanmedian(env[mask])
    if not np.isfinite(noise) or noise <= 0:
        return t, np.full_like(env, np.nan)
    return t, env / noise


def shared_arrays(tr_a, tr_b, max_lag_s=0.25):
    sr = float(tr_a.stats.sampling_rate)
    if not np.isclose(sr, float(tr_b.stats.sampling_rate)):
        raise ValueError("Sampling rates differ")

    start = max(tr_a.stats.starttime, tr_b.stats.starttime)
    end = min(tr_a.stats.endtime, tr_b.stats.endtime)
    a = tr_a.copy().trim(start, end).data.astype(float)
    b = tr_b.copy().trim(start, end).data.astype(float)
    n = min(len(a), len(b))
    a, b = a[:n], b[:n]
    good = np.isfinite(a) & np.isfinite(b)
    a, b = a[good], b[good]

    a0, b0 = a - a.mean(), b - b.mean()
    nlag = int(round(max_lag_s * sr))
    lags = np.arange(-nlag, nlag + 1)
    scores = []
    for lag in lags:
        if lag > 0:
            aa, bb = a0[lag:], b0[:-lag]
        elif lag < 0:
            aa, bb = a0[:lag], b0[-lag:]
        else:
            aa, bb = a0, b0
        scores.append(np.dot(aa, bb))
    lag = int(lags[np.argmax(scores)])

    if lag > 0:
        aa, bb = a[lag:], b[:-lag]
    elif lag < 0:
        aa, bb = a[:lag], b[-lag:]
    else:
        aa, bb = a, b
    return aa, bb, lag / sr


def pair_metrics(tr_a, tr_b):
    a, b, lag_s = shared_arrays(tr_a, tr_b)
    rms = lambda x: np.sqrt(np.mean(x * x))
    return {
        "rms_ratio": rms(a) / rms(b),
        "lag_s": lag_s,
        "lagged_correlation": np.corrcoef(a, b)[0, 1],
        "npts": len(a),
    }

In [3]:
def read_raw_event(row):
    """Read portable extract, or create it from the configured local SDS."""
    path = INPUT_DIR / row.file
    if path.is_file():
        return read(str(path))

    if config.SDS_DIR is None or not Path(config.SDS_DIR).is_dir():
        raise FileNotFoundError(
            f"Missing {path}. Put the raw-count extract there, or configure "
            "external_paths.sds_dir in local_config.toml."
        )

    t0 = UTCDateTime(row.event_time_utc)
    client = SDSClient(str(config.SDS_DIR))
    st = Stream()
    for channel in ("DD1", "DD2", "DD3"):
        st += client.get_waveforms(
            "1R", "BCHH", "10", channel, t0 - PRE_S, t0 + POST_S
        )
    if not st:
        raise FileNotFoundError(
            f"No BCHH pressure data found in {config.SDS_DIR} for {row.event_id}."
        )

    INPUT_DIR.mkdir(parents=True, exist_ok=True)
    st.write(str(path), format="MSEED")
    print("Created portable raw-count extract:", path)
    return st


def read_and_window(row):
    t0 = UTCDateTime(row.event_time_utc)
    st = preprocess_counts(
        read_raw_event(row).trim(t0 - PRE_S, t0 + POST_S)
    )
    by_id = {tr.id: tr for tr in st}
    if any(seed not in by_id for seed in PRESSURE_IDS[1:]):
        raise ValueError(f"{row.event_id} is missing DD2 or DD3")

    onsets = []
    for seed in PRESSURE_IDS[1:]:
        t, snr = channel_snr(by_id[seed], t0)
        hits = np.flatnonzero(
            (t >= SEARCH_WINDOW[0])
            & (t <= SEARCH_WINDOW[1])
            & (snr >= ONSET_SNR)
        )
        if not len(hits):
            raise ValueError(
                f"No DD2/DD3 SNR>{ONSET_SNR} onset for {row.event_id}; "
                "inspect the waveform before changing thresholds."
            )
        onsets.append(float(t[hits[0]]))

    start = max(onsets)
    stop = min(start + CAL_WINDOW_LENGTH_S, SEARCH_WINDOW[1])
    traces = {
        seed: by_id[seed].copy().trim(t0 + start, t0 + stop)
        for seed in by_id
    }
    return st, traces, {
        "event_id": row.event_id,
        "mission": row.mission,
        "role": row.role,
        "signal_start_s": start,
        "signal_end_s": stop,
    }


streams, windows, qc_rows = {}, {}, []
for row in EVENTS.itertuples(index=False):
    st, traces, qc = read_and_window(row)
    streams[row.event_id] = st
    windows[row.event_id] = traces
    qc_rows.append(qc)

window_qc = pd.DataFrame(qc_rows)
display(window_qc)

FileNotFoundError: Missing /Users/thompsong/Library/CloudStorage/Box-Box/thompsong/3_Project_Documents/NASAprojects/201602_Rocket_Seismology/writing/falcon9_fireball_paper/inputs/miniseed/2016_launch_calibration/jcsat16_bchh_raw.mseed. Put the raw-count extract there, or configure external_paths.sds_dir in local_config.toml.

In [ ]:
metrics = []
for row in EVENTS.itertuples(index=False):
    traces = windows[row.event_id]
    for left, right in combinations(PRESSURE_IDS, 2):
        if left not in traces or right not in traces:
            continue
        result = pair_metrics(traces[left], traces[right])
        metrics.append({
            "event_id": row.event_id,
            "mission": row.mission,
            "role": row.role,
            "pair": f"{left.split('.')[-1]}/{right.split('.')[-1]}",
            **result,
        })

relative = pd.DataFrame(metrics)
relative["expected_lift_ratio"] = relative["pair"].map({
    "DD1/DD2": ADOPTED_COUNTS_PER_PA["DD1"] / ADOPTED_COUNTS_PER_PA["DD2"],
    "DD1/DD3": ADOPTED_COUNTS_PER_PA["DD1"] / ADOPTED_COUNTS_PER_PA["DD3"],
    "DD2/DD3": ADOPTED_COUNTS_PER_PA["DD2"] / ADOPTED_COUNTS_PER_PA["DD3"],
})
relative["interpretation"] = np.where(
    relative["pair"].eq("DD2/DD3"),
    "primary relative-gain comparison",
    "secondary QC; DD1 is noisier",
)
display(relative)

fig, ax = plt.subplots(figsize=(8.5, 4.8))
for pair, group in relative.groupby("pair", sort=False):
    ax.plot(group["mission"], group["rms_ratio"], "o-", label=pair)
for pair, expected in relative.groupby("pair")["expected_lift_ratio"].first().items():
    ax.axhline(expected, lw=0.9, ls="--", color="0.5")
ax.set_ylabel("Filtered raw-count RMS ratio")
ax.set_title("BCHH relative-gain validation; dashed = lift-test prediction")
ax.tick_params(axis="x", rotation=20)
ax.legend()
fig.tight_layout()
fig.savefig(OUT / "relative_gain_filtered_counts.pdf", bbox_inches="tight")
plt.show()

## Interpretation guardrails

The primary comparison is **DD2/DD3**. The routine launches should be inspected for clustering around the lift-test prediction of 0.0690. AMOS-6 is an extreme-amplitude stress test and must not be used to tune the adopted conversion factors.

This notebook can support a manuscript statement that the **relative gain was stable across nearby 2016 events** if the plotted/resulting routine-launch ratios support that statement. It must not be cited as evidence for absolute calibration accuracy or sensor linearity at the AMOS-6 peak.

In [ ]:
window_qc.to_csv(OUT / "signal_window_qc.csv", index=False)
relative.to_csv(OUT / "relative_gain_filtered_counts.csv", index=False)
EVENTS.to_csv(OUT / "event_provenance.csv", index=False)

pd.DataFrame([{
    "adopted_dd1_counts_per_pa": ADOPTED_COUNTS_PER_PA["DD1"],
    "adopted_dd2_counts_per_pa": ADOPTED_COUNTS_PER_PA["DD2"],
    "adopted_dd3_counts_per_pa": ADOPTED_COUNTS_PER_PA["DD3"],
    "dd2_dd3_lift_prediction": (
        ADOPTED_COUNTS_PER_PA["DD2"] / ADOPTED_COUNTS_PER_PA["DD3"]
    ),
}]).to_csv(OUT / "calibration_reference_values.csv", index=False)

print("Wrote calibration validation products to", OUT)